# <font color='lightgreen'>02 clean data  🧹</font>

	Aplicamos reglas de calidad, corregimos valores fuera de rango, manejamos nulos y outliers, y generamos un reporte de calidad.
	Este notebook es **idempotente** y **auditable**: registra cada transformación.
---

### <font color='lightgreen'>Librerías</font>

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import json
import warnings
warnings.filterwarnings('ignore')
print("Librerías importadas correctamente")
print('Versión de pandas:', pd.__version__)
print('Versión de numpy:', np.__version__)


Librerías importadas correctamente
Versión de pandas: 3.0.2
Versión de numpy: 2.4.4


### <font color='lightgreen'>Configuración</font>

#### <font color='lightgreen'>Configuración logging</font>

In [10]:
# Configurar logging
logging.basicConfig(
	level=logging.INFO,
	format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

#### <font color='lightgreen'>Configuración de rutas</font>

Usamos `pathlib` para rutas consistentes, basadas en la raíz del proyecto.

In [11]:
# Raíz del proyecto (asumiendo que el notebook está en notebooks/)
PROJECT_ROOT = Path.cwd().parent
DATA_LOADED = PROJECT_ROOT / "data" / "loaded"
DATA_CLEAN = PROJECT_ROOT / "data" / "clean"

# Crear carpeta de salida si no existe
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

logger.info(f"📁 Datos sucios: {DATA_LOADED}")
logger.info(f"📁 Datos limpios: {DATA_CLEAN}")

2026-04-15 18:08:32,707 - INFO - 📁 Datos sucios: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\loaded
2026-04-15 18:08:32,724 - INFO - 📁 Datos limpios: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\clean


### <font color='lightgreen'>Funciones auxiliares</font>

Definimos herramientas reutilizables:
- `detectar_outliers`: identifica valores atípicos usando IQR.
- `corregir_outliers`: los recorta (clip) o elimina.
- `resumen_calidad`: genera estadísticas de calidad de un DataFrame.

In [12]:
def validar_rango_fechas(df, columna, año_minimo=2000):
    """
    Convierte a NaT las fechas anteriores al año mínimo (ej. 2000).
    Esto elimina fechas inválidas como 1970-01-01.
    """
    if columna in df.columns and pd.api.types.is_datetime64_any_dtype(df[columna]):
        fecha_minima = pd.Timestamp(f'{año_minimo}-01-01')
        antes = df[columna].isna().sum()
        df[columna] = df[columna].where(df[columna] >= fecha_minima, pd.NaT)
        despues = df[columna].isna().sum()
        if despues > antes:
            logger.info(f"   🗓️ {columna}: {despues - antes} fechas anteriores a {año_minimo} convertidas a NaT")
    return df

def detectar_outliers(df, columnas_numericas):
    """
    Detecta outliers en columnas numéricas usando el rango intercuartil (IQR).
    
    Retorna: diccionario {columna: cantidad_outliers}
    """
    outliers = {}
    for col in columnas_numericas:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers_count = ((df[col] < lower) | (df[col] > upper)).sum()
            if outliers_count > 0:
                outliers[col] = int(outliers_count)
    return outliers

def corregir_outliers(df, col, metodo='clip'):
    """
    Corrige outliers en una columna numérica.
    metodo: 'clip' -> limita al rango [Q1-1.5IQR, Q3+1.5IQR]
            'remove' -> elimina filas con outliers
    Retorna: DataFrame modificado
    """
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    if metodo == 'clip':
        antes = len(df)
        df[col] = df[col].clip(lower, upper)
        logger.info(f"   ✂️ {col}: {antes - len(df)} valores recortados")
    elif metodo == 'remove':
        antes = len(df)
        df = df[(df[col] >= lower) & (df[col] <= upper)]
        logger.info(f"   🗑️ {col}: se eliminaron {antes - len(df)} filas con outliers")
    return df

def resumen_calidad(df, nombre, etapa):
    """
    Genera un resumen de calidad: nulos, duplicados, outliers.
    Retorna un diccionario con las métricas.
    """
    reporte = {
        "nombre": nombre,
        "etapa": etapa,
        "filas": len(df),
        "columnas": len(df.columns),
        "nulos_por_columna": df.isnull().sum().to_dict(),
        "duplicados": int(df.duplicated().sum()),
        "outliers": detectar_outliers(df, df.select_dtypes(include='number').columns.tolist())
    }
    return reporte

### <font color='lightgreen'>Cargar datos raw</font>

Leemos todos los archivos generados en el paso anterior.

In [13]:
archivos = [
    "compras",
    "consumo_energetico",
    "encuestas",
    "eventos_rrhh",
    "personal_nomina",
    "residuos",
    "ventas"
]

datos_clean = {}   # Diccionario para guardar los DataFrames limpios
reportes = []      # Lista para acumular reportes de calidad

logger.info("="*50)
logger.info("CARGANDO DATOS RAW")
logger.info("="*50)

for nombre in archivos:
    file_path = DATA_LOADED / f"{nombre}_raw.csv"
    if file_path.exists():
        df = pd.read_csv(file_path, encoding='utf-8')
        datos_clean[nombre] = df
        logger.info(f"✅ {nombre}: {len(df):,} filas, {len(df.columns)} columnas")
        
        # Guardar reporte inicial
        reportes.append(resumen_calidad(df, nombre, "raw"))
    else:
        logger.error(f"❌ No se encuentra {file_path}")

2026-04-15 18:08:33,152 - INFO - ==================================================
2026-04-15 18:08:33,154 - INFO - CARGANDO DATOS RAW
2026-04-15 18:08:33,156 - INFO - ==================================================
2026-04-15 18:08:33,165 - INFO - ✅ compras: 864 filas, 8 columnas
2026-04-15 18:08:33,188 - INFO - ✅ consumo_energetico: 178 filas, 12 columnas
2026-04-15 18:08:33,212 - INFO - ✅ encuestas: 189 filas, 12 columnas
2026-04-15 18:08:33,233 - INFO - ✅ eventos_rrhh: 537 filas, 10 columnas
2026-04-15 18:08:33,258 - INFO - ✅ personal_nomina: 1,378 filas, 12 columnas
2026-04-15 18:08:33,290 - INFO - ✅ residuos: 2,508 filas, 10 columnas
2026-04-15 18:08:33,493 - INFO - ✅ ventas: 43,893 filas, 15 columnas


#### <font color='lightgreen'>Normalización formato fecha</font></font>

In [14]:
# Después de cargar y convertir fechas en cada dataset
if 'fecha' in df.columns:
    # Reemplazar fechas anteriores al año 2000 por NaT
    df['fecha'] = df['fecha'].where(df['fecha'] >= '2000-01-01', pd.NaT)
    logger.info(f"   Fechas inválidas (antes de 2000) convertidas a NaT")

2026-04-15 18:08:33,634 - INFO -    Fechas inválidas (antes de 2000) convertidas a NaT


### <font color='lightgreen'>Aplicar limpieza específica por cada dataset</font>

Cada dataset tiene reglas de calidad particulares. Las aplicamos de forma clara y documentada.

#### **1.- Compras**
- Validamos que `cantidad` > 0 y `precio_unitario` > 0

In [15]:
nombre = "compras"
df = datos_clean[nombre].copy()
logger.info(f"\n📊 Limpiando {nombre}...")

# Eliminar duplicados
antes = len(df)
df = df.drop_duplicates()
logger.info(f"   Duplicados eliminados: {antes - len(df)}")

# Montos no negativos
if 'monto_ars' in df.columns:
    negativos = (df['monto_ars'] < 0).sum()
    if negativos > 0:
        df['monto_ars'] = df['monto_ars'].clip(lower=0)
        logger.info(f"   Montos negativos corregidos: {negativos}")

# Fechas: ya están como datetime, pero verificamos nulos
if 'fecha_emision' in df.columns:
    nulos_fecha = df['fecha_emision'].isna().sum()
    if nulos_fecha > 0:
        logger.warning(f"   {nulos_fecha} fechas de emisión nulas (se mantienen)")

datos_clean[nombre] = df

# Después de cargar df y antes de guardar
for col in ['fecha_emision', 'fecha_pago']:
    df = validar_rango_fechas(df, col)
reportes.append(resumen_calidad(df, nombre, "clean"))

2026-04-15 18:08:33,659 - INFO - 
📊 Limpiando compras...
2026-04-15 18:08:33,664 - INFO -    Duplicados eliminados: 0


#### **2.- Consumo Energético**
- Validamos que `consumo_kwh` > 0

In [16]:
nombre = "consumo_energetico"
df = datos_clean[nombre].copy()
logger.info(f"\n📊 Limpiando {nombre}...")

# Duplicados
antes = len(df)
df = df.drop_duplicates()
logger.info(f"   Duplicados eliminados: {antes - len(df)}")

# Consumos no negativos
for col in ['kwh_consumidos', 'm3_gas']:
    if col in df.columns:
        negativos = (df[col] < 0).sum()
        if negativos > 0:
            df[col] = df[col].clip(lower=0)
            logger.info(f"   {col} negativos corregidos: {negativos}")

# Recalcular kwh_totales si existe la columna
if 'kwh_totales' in df.columns:
    df['kwh_totales'] = df['kwh_consumidos'].fillna(0) + (df['m3_gas'].fillna(0) * 10.69)
    logger.info("   kwh_totales recalculado")

datos_clean[nombre] = df
# Después de cargar df y antes de guardar
for col in ['fecha']:
    df = validar_rango_fechas(df, col)
reportes.append(resumen_calidad(df, nombre, "clean"))

2026-04-15 18:08:33,698 - INFO - 
📊 Limpiando consumo_energetico...
2026-04-15 18:08:33,702 - INFO -    Duplicados eliminados: 0


#### **3.- Encuestas clima**
- Validamos que `satisfaccion` esté entre 1 y 5

In [17]:
nombre = "encuestas"
df = datos_clean[nombre].copy()
logger.info(f"\n📊 Limpiando {nombre}...")

# Duplicados
antes = len(df)
df = df.drop_duplicates()
logger.info(f"   Duplicados eliminados: {antes - len(df)}")

# Validar rangos de satisfacción (1-10)
satisfaccion_cols = ['satisfaccion_gral', 'clima_equipo', 'liderazgo', 'condiciones_fisicas']
for col in satisfaccion_cols:
    if col in df.columns:
        fuera_rango = ((df[col] < 1) | (df[col] > 10)).sum()
        if fuera_rango > 0:
            df[col] = df[col].clip(1, 10)
            logger.info(f"   {col}: {fuera_rango} valores fuera de rango corregidos")

datos_clean[nombre] = df
# Después de cargar df y antes de guardar
for col in ['fecha_encuesta']:
    df = validar_rango_fechas(df, col)
reportes.append(resumen_calidad(df, nombre, "clean"))

2026-04-15 18:08:33,752 - INFO - 
📊 Limpiando encuestas...
2026-04-15 18:08:33,758 - INFO -    Duplicados eliminados: 0


#### **4.- Eventos RRHH**
- Validamos que `duracion_horas` > 0

In [18]:
nombre = "eventos_rrhh"
df = datos_clean[nombre].copy()
logger.info(f"\n📊 Limpiando {nombre}...")

# Duplicados
antes = len(df)
df = df.drop_duplicates()
logger.info(f"   Duplicados eliminados: {antes - len(df)}")

# Horas capacitación y días baja no negativos
for col in ['horas_capacitacion', 'dias_baja']:
    if col in df.columns:
        negativos = (df[col] < 0).sum()
        if negativos > 0:
            df[col] = df[col].clip(lower=0)
            logger.info(f"   {col} negativos corregidos: {negativos}")

datos_clean[nombre] = df
# Después de cargar df y antes de guardar
for col in ['fecha']:
    df = validar_rango_fechas(df, col)
reportes.append(resumen_calidad(df, nombre, "clean"))

2026-04-15 18:08:33,829 - INFO - 
📊 Limpiando eventos_rrhh...
2026-04-15 18:08:33,833 - INFO -    Duplicados eliminados: 0


#### **5.- Personal Nómina**
- Validamos que `salario` > 0

In [19]:
nombre = "personal_nomina"
df = datos_clean[nombre].copy()
logger.info(f"\n📊 Limpiando {nombre}...")

# Duplicados
antes = len(df)
df = df.drop_duplicates()
logger.info(f"   Duplicados eliminados: {antes - len(df)}")

# Salarios no negativos
if 'total_devengado' in df.columns:
    negativos = (df['total_devengado'] < 0).sum()
    if negativos > 0:
        df['total_devengado'] = df['total_devengado'].clip(lower=0)
        logger.info(f"   Salarios negativos corregidos: {negativos}")

datos_clean[nombre] = df
# Después de cargar df y antes de guardar
for col in ['fecha_ingreso', 'fecha']:
    df = validar_rango_fechas(df, col)
reportes.append(resumen_calidad(df, nombre, "clean"))

2026-04-15 18:08:33,876 - INFO - 
📊 Limpiando personal_nomina...
2026-04-15 18:08:33,885 - INFO -    Duplicados eliminados: 0


#### **6.- Residuos reciclaje**
- Validamos que `cantidad_kg` > 0

In [20]:
nombre = "residuos"
df = datos_clean[nombre].copy()
logger.info(f"\n📊 Limpiando {nombre}...")

# Duplicados
antes = len(df)
df = df.drop_duplicates()
logger.info(f"   Duplicados eliminados: {antes - len(df)}")

# Kilogramos no negativos
for col in ['kg_generados', 'kg_reciclados']:
    if col in df.columns:
        negativos = (df[col] < 0).sum()
        if negativos > 0:
            df[col] = df[col].clip(lower=0)
            logger.info(f"   {col} negativos corregidos: {negativos}")

# Recalcular tasa_reciclaje
if 'kg_reciclados' in df.columns and 'kg_generados' in df.columns:
    # Evitar división por cero
    df['tasa_reciclaje'] = (df['kg_reciclados'] / df['kg_generados'].replace(0, np.nan)) * 100
    df['tasa_reciclaje'] = df['tasa_reciclaje'].fillna(0).clip(0, 100)
    logger.info("   tasa_reciclaje recalculada")

datos_clean[nombre] = df
# Después de cargar df y antes de guardar
for col in ['fecha_semana']:
    df = validar_rango_fechas(df, col)
reportes.append(resumen_calidad(df, nombre, "clean"))

2026-04-15 18:08:33,924 - INFO - 
📊 Limpiando residuos...
2026-04-15 18:08:33,932 - INFO -    Duplicados eliminados: 0
2026-04-15 18:08:33,968 - INFO -    tasa_reciclaje recalculada


#### **7.- Ventas**
- Validamos que `cantidad` > 0 y `precio_unitario` > 0

In [21]:
nombre = "ventas"
df = datos_clean[nombre].copy()
logger.info(f"\n📊 Limpiando {nombre}...")

# Duplicados
antes = len(df)
df = df.drop_duplicates()
logger.info(f"   Duplicados eliminados: {antes - len(df)}")

# Cantidades positivas (<=0 no tienen sentido)
if 'cantidad' in df.columns:
    no_positivas = (df['cantidad'] <= 0).sum()
    if no_positivas > 0:
        df = df[df['cantidad'] > 0]
        logger.info(f"   Eliminadas {no_positivas} filas con cantidad <= 0")

# Precios y subtotales no negativos
for col in ['precio_unitario', 'subtotal_ars']:
    if col in df.columns:
        negativos = (df[col] < 0).sum()
        if negativos > 0:
            df[col] = df[col].clip(lower=0)
            logger.info(f"   {col} negativos corregidos: {negativos}")

datos_clean[nombre] = df
# Después de cargar df y antes de guardar
for col in ['fecha']:
    df = validar_rango_fechas(df, col)
reportes.append(resumen_calidad(df, nombre, "clean"))

2026-04-15 18:08:34,025 - INFO - 
📊 Limpiando ventas...
2026-04-15 18:08:34,105 - INFO -    Duplicados eliminados: 0


### <font color='lightgreen'>Tratamiento de outliers (opcional)</font>

Decidimos **no eliminar** outliers, sino recortarlos (clip) para no perder información.
Aplicamos solo a columnas críticas.

In [22]:
logger.info("\n" + "="*50)
logger.info("TRATAMIENTO DE OUTLIERS (CLIP)")
logger.info("="*50)

# Definir columnas a tratar por cada dataset
outlier_config = {
    "compras": ["monto_ars"],
    "ventas": ["subtotal_ars", "cantidad"],
    "consumo_energetico": ["kwh_consumidos", "m3_gas"],
    "residuos": ["kg_generados", "kg_reciclados"]
}

for nombre, columnas in outlier_config.items():
    if nombre in datos_clean:
        df = datos_clean[nombre]
        for col in columnas:
            if col in df.columns:
                df = corregir_outliers(df, col, metodo='clip')
        datos_clean[nombre] = df
        # Actualizar reporte después de corregir outliers
        reportes.append(resumen_calidad(df, nombre, "clean_outliers"))

2026-04-15 18:08:34,226 - INFO - 
2026-04-15 18:08:34,227 - INFO - TRATAMIENTO DE OUTLIERS (CLIP)
2026-04-15 18:08:34,228 - INFO - ==================================================
2026-04-15 18:08:34,239 - INFO -    ✂️ monto_ars: 0 valores recortados
2026-04-15 18:08:34,256 - INFO -    ✂️ subtotal_ars: 0 valores recortados
2026-04-15 18:08:34,265 - INFO -    ✂️ cantidad: 0 valores recortados
2026-04-15 18:08:34,365 - INFO -    ✂️ kwh_consumidos: 0 valores recortados
2026-04-15 18:08:34,370 - INFO -    ✂️ m3_gas: 0 valores recortados
2026-04-15 18:08:34,391 - INFO -    ✂️ kg_generados: 0 valores recortados
2026-04-15 18:08:34,396 - INFO -    ✂️ kg_reciclados: 0 valores recortados


### <font color='lightgreen'>Guardar archivos limpios</font>

In [23]:
logger.info("\n" + "="*50)
logger.info("GUARDANDO ARCHIVOS LIMPIOS")
logger.info("="*50)

for nombre, df in datos_clean.items():
    output_file = DATA_CLEAN / f"{nombre}_clean.csv"
    df.to_csv(output_file, index=False, encoding='utf-8')
    logger.info(f"💾 {nombre}_clean.csv ({len(df):,} filas, {len(df.columns)} columnas)")

2026-04-15 18:08:34,430 - INFO - 
2026-04-15 18:08:34,431 - INFO - GUARDANDO ARCHIVOS LIMPIOS
2026-04-15 18:08:34,433 - INFO - ==================================================
2026-04-15 18:08:34,517 - INFO - 💾 compras_clean.csv (864 filas, 8 columnas)
2026-04-15 18:08:34,524 - INFO - 💾 consumo_energetico_clean.csv (178 filas, 12 columnas)
2026-04-15 18:08:34,531 - INFO - 💾 encuestas_clean.csv (189 filas, 12 columnas)
2026-04-15 18:08:34,538 - INFO - 💾 eventos_rrhh_clean.csv (537 filas, 10 columnas)
2026-04-15 18:08:34,554 - INFO - 💾 personal_nomina_clean.csv (1,378 filas, 12 columnas)
2026-04-15 18:08:34,583 - INFO - 💾 residuos_clean.csv (2,508 filas, 11 columnas)
2026-04-15 18:08:35,104 - INFO - 💾 ventas_clean.csv (43,893 filas, 15 columnas)


### <font color='lightgreen'>Guardar reporte de calidad</font>

Exportamos el reporte completo a JSON para auditoría.

In [24]:
reporte_path = DATA_CLEAN / "calidad_report.json"
with open(reporte_path, "w", encoding='utf-8') as f:
    json.dump(reportes, f, indent=2, default=str, ensure_ascii=False)

logger.info(f"📊 Reporte de calidad guardado en {reporte_path}")

2026-04-15 18:08:36,068 - INFO - 📊 Reporte de calidad guardado en c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\clean\calidad_report.json


### <font color='lightgreen'>Resumen final</font>

In [25]:
print("\n" + "="*50)
print("✅ LIMPIEZA COMPLETADA")
print("="*50)

print("\n📊 Resumen de datasets limpios:")
print("-"*60)
print(f"{'Dataset':<20} {'Filas':>10} {'Columnas':>10} {'Nulos total':>12}")
print("-"*60)

for nombre, df in datos_clean.items():
    nulos_total = df.isnull().sum().sum()
    print(f"{nombre:<20} {len(df):>10,} {len(df.columns):>10} {nulos_total:>12,}")

print(f"\n📁 Archivos guardados en: {DATA_CLEAN}/")
print(f"📄 Reporte de calidad: {reporte_path}")
print("\n🔍 Siguiente paso: Ejecutar 03_build_star_schema.ipynb")


✅ LIMPIEZA COMPLETADA

📊 Resumen de datasets limpios:
------------------------------------------------------------
Dataset                   Filas   Columnas  Nulos total
------------------------------------------------------------
compras                     864          8            0
consumo_energetico          178         12            0
encuestas                   189         12            0
eventos_rrhh                537         10            0
personal_nomina           1,378         12            0
residuos                  2,508         11            0
ventas                   43,893         15            0

📁 Archivos guardados en: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\clean/
📄 Reporte de calidad: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Inte